# Homework 1 Final Submission



In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from final_submission_utils import (
    PHASE2_TARGET_DATES,
    ROOT,
    analyze_signal_consistency,
    build_phase1_summary,
    build_rule_compliant_replay,
    build_submission_trade_log,
    device,
    ensure_artifact_dir,
    evaluate_final_model,
    load_actual_phase3_submission,
    load_base_history,
    load_extended_history,
    phase2_metrics,
    run_phase2_rolling_forecast,
    save_asset_curve_plot,
    save_phase1_metric_plot,
    save_phase3_drawdown_comparison_plot,
    save_phase3_position_breakdown_plot,
    save_phase3_signal_return_plot,
    save_prediction_plot,
    write_report,
)

artifact_dir = ensure_artifact_dir()
report_output_path = ROOT / "Homework1_report.md"
print(f"Artifact directory: {artifact_dir}")
print(f"Current device: {device()}")

Artifact directory: /ssd6/jiakai/114_2_RNN/final_submission_artifacts
Current device: cuda


## Phase 1: Baseline And Tuning Summary

這一段直接整理 repo 中已經完成的代表性實驗，覆蓋：
- baseline sample code
- sequence length 調整
- model architecture / dropout
- training parameters
- feature engineering
- loss function

In [2]:
phase1_df = build_phase1_summary()
phase1_path = artifact_dir / "phase1_summary.csv"
phase1_df.to_csv(phase1_path, index=False)
phase1_metric_plot_path = artifact_dir / "phase1_metrics.png"
save_phase1_metric_plot(phase1_df, phase1_metric_plot_path)
phase1_df.round(4)

,Model,Category,Adjustment,Test RMSE,Test MAPE (%),RMSE Improvement vs Baseline,MAPE Improvement vs Baseline
0,Baseline sample code,Provided notebook,Original Stock_predict.ipynb default setting,60.9182,3.03,0.0000,0.00
1,v2_lb60,Sequence length,look_back 100 -> 60,49.9911,2.33,10.9271,0.70
2,v3_deep_dropout,Architecture + Dropout,"hidden [256,128,64], dropout 0.2",456.2289,26.04,-395.3107,-23.01
3,v4_train_tune,Training params,"lr 0.0005, batch 64, epochs 80",62.5129,2.98,-1.5947,0.05
4,v5_indicators,Feature engineering,Close + MA5 + MA20 + RSI14,82.5861,4.25,-21.6679,-1.22
5,v7_lb50,Sequence length,look_back 50,38.7006,1.81,22.2176,1.22
6,v14_lb67,Sequence length,look_back 67,42.1649,1.96,18.7533,1.07
7,v16_lb67_smoothl1,Loss function,look_back 67 + SmoothL1Loss,32.9998,1.57,27.9184,1.46


In [3]:
display(Markdown(f"![Phase 1 Metrics]({phase1_metric_plot_path.relative_to(ROOT)})"))

![Phase 1 Metrics](final_submission_artifacts/phase1_metrics.png)

## Final Model Reproduction

最終模型採用 `v16_lb67_smoothl1`：
- `look_back=67`
- hidden sizes = `[128, 64]`
- `SmoothL1Loss(beta=0.05)`
- `lr=0.001`, `batch_size=32`, `epochs=50`

下方會在 `2330_stock_data.csv` 上重新訓練一次，確認最終模型在 homework 的 hold-out split 上的表現。

In [4]:
base_df = load_base_history()
final_eval = evaluate_final_model(base_df)
final_metrics = pd.DataFrame([final_eval["metrics"]]).round(4)
final_prediction_path = artifact_dir / "phase1_final_model_predictions.csv"
final_eval["prediction_df"].to_csv(final_prediction_path, index=False)

phase1_plot_path = artifact_dir / "phase1_test_prediction.png"
save_prediction_plot(
    final_eval["prediction_df"],
    phase1_plot_path,
    "Phase 1 Final Model: Hold-out Test Prediction vs Actual",
)

final_metrics

,train_rmse,train_mae,train_mape,test_rmse,test_mae,test_mape
0,13.3568,9.3508,2.2984,32.9998,25.124,1.5707


In [5]:
display(Markdown(f"![Phase 1 Test Prediction]({phase1_plot_path.relative_to(ROOT)})"))

![Phase 1 Test Prediction](final_submission_artifacts/phase1_test_prediction.png)

## Phase 2: Rolling Forecast

選擇 **2026-03-06 到 2026-03-19** 這 10 個交易日做 rolling forecast。

流程：
1. 用目標日之前的所有資料訓練模型。
2. 只預測下一個交易日。
3. 觀察真實收盤價後，把新資料加入訓練集合。
4. 進行 daily update，再往下一天預測。

In [6]:
extended_df = load_extended_history()
close_series = extended_df["Close"].copy()

rolling_df = run_phase2_rolling_forecast(close_series, target_dates=PHASE2_TARGET_DATES)
rolling_metric_values = phase2_metrics(rolling_df)
rolling_path = artifact_dir / "phase2_rolling_forecast.csv"
rolling_df.to_csv(rolling_path, index=False)

rolling_plot_path = artifact_dir / "phase2_rolling_forecast.png"
save_prediction_plot(
    rolling_df,
    rolling_plot_path,
    "Phase 2 Rolling Forecast: Prediction vs Actual",
)

print(
    f"Phase 2 RMSE: {rolling_metric_values['rmse']:.4f}, "
    f"MAPE: {rolling_metric_values['mape']:.2f}%"
)
rolling_df.round(4)

Phase 2 RMSE: 59.0017, MAPE: 2.70%


,Date,PrevClose,Predicted,Actual,PredReturnPct,ActualReturnPct,AbsError,EpochsUsed
0,2026-03-06 00:00:00+08:00,1900.0,1960.0239,1890.0,3.1592,-0.5263,70.0239,50
1,2026-03-09 00:00:00+08:00,1890.0,1897.1674,1810.0,0.3792,-4.2328,87.1674,10
2,2026-03-10 00:00:00+08:00,1810.0,1839.9779,1850.0,1.6562,2.2099,10.0221,10
3,2026-03-11 00:00:00+08:00,1850.0,1843.6584,1940.0,-0.3428,4.8649,96.3416,10
4,2026-03-12 00:00:00+08:00,1940.0,1965.1437,1885.0,1.2961,-2.8351,80.1437,10
5,2026-03-13 00:00:00+08:00,1885.0,1868.7180,1865.0,-0.8638,-1.0610,3.7180,10
6,2026-03-16 00:00:00+08:00,1865.0,1890.9083,1845.0,1.3892,-1.0724,45.9083,10
7,2026-03-17 00:00:00+08:00,1845.0,1834.1260,1870.0,-0.5894,1.3550,35.8740,10
8,2026-03-18 00:00:00+08:00,1870.0,1859.9512,1905.0,-0.5374,1.8717,45.0488,10
9,2026-03-19 00:00:00+08:00,1905.0,1882.6753,1850.0,-1.1719,-2.8871,32.6753,10


In [7]:
display(Markdown(f"![Phase 2 Rolling Forecast]({rolling_plot_path.relative_to(ROOT)})"))

![Phase 2 Rolling Forecast](final_submission_artifacts/phase2_rolling_forecast.png)

## Phase 3: Actual Submission Analysis And Rule-Compliant Replay

本段使用 `2408.TW` 的最終交易紀錄。

這裡分成三個層次：
1. 先整理原始提交內容。
2. 檢查模型預測方向與實際操作是否一致。
3. 產生兩份績效表：
   - 依預測價格記帳的提交版本
   - 依收盤價結算且符合作業規則的合法 replay

In [8]:
actual_submission_df = load_actual_phase3_submission()
day4_mask = actual_submission_df["Sheet"] == "Day 4"
actual_submission_df.loc[day4_mask, "SubmittedQuantity"] = 12000
holdings_before_day10 = 0
for _, row in actual_submission_df.iterrows():
    if row["Sheet"] == "Day 10":
        break
    if row["SubmittedAction"] == "Buy":
        holdings_before_day10 += int(row["SubmittedQuantity"])
    elif row["SubmittedAction"] == "Sell":
        holdings_before_day10 -= int(row["SubmittedQuantity"])

day10_mask = actual_submission_df["Sheet"] == "Day 10"
actual_submission_df.loc[day10_mask, "SubmittedQuantity"] = holdings_before_day10

actual_submission_path = artifact_dir / "phase3_submission_input.csv"
actual_submission_df.to_csv(actual_submission_path, index=False)
actual_submission_df[
    [
        "Date",
        "Sheet",
        "SubmittedAction",
        "SubmittedQuantity",
        "PredictedPrice",
        "CurrentClose",
        "NextClose",
    ]
].round(4)

,Date,Sheet,SubmittedAction,SubmittedQuantity,PredictedPrice,CurrentClose,NextClose
0,2026-03-20,Day 1,Buy,8547,226.98,234.0,223.0
1,2026-03-23,Day 2,Buy,13343,224.83,223.0,216.5
2,2026-03-24,Day 3,Buy,10000,222.93,216.5,226.5
3,2026-03-25,Day 4,Buy,12000,208.37,226.5,225.5
4,2026-03-26,Day 5,Hold,0,220.37,225.5,219.5
5,2026-03-27,Day 6,Hold,0,214.23,219.5,220.5
6,2026-03-30,Day 7,Hold,0,206.94,220.5,198.5
7,2026-03-31,Day 8,Hold,0,202.49,198.5,211.0
8,2026-04-01,Day 9,Hold,0,214.70,211.0,200.5
9,2026-04-02,Day 10,Sell,43890,220.00,200.5,NaN


In [9]:
consistency_df, consistency_summary = analyze_signal_consistency(actual_submission_df)
consistency_path = artifact_dir / "phase3_signal_consistency.csv"
consistency_df.to_csv(consistency_path, index=False)
phase3_signal_plot_path = artifact_dir / "phase3_signal_return_comparison.png"
save_phase3_signal_return_plot(consistency_df, phase3_signal_plot_path)

print(
    f"Comparable days: {consistency_summary['comparable_count']}, "
    f"Consistent days: {consistency_summary['consistent_count']}, "
    f"Consistency rate: {consistency_summary['consistency_rate_pct']:.2f}%"
)
consistency_df[
    ["Date", "PredictedPrice", "CurrentClose", "PredictedDeltaVsCurrentPct", "ImpliedAction", "SubmittedAction", "Consistent"]
].round(4)

Comparable days: 9, Consistent days: 2, Consistency rate: 22.22%


,Date,PredictedPrice,CurrentClose,PredictedDeltaVsCurrentPct,ImpliedAction,SubmittedAction,Consistent
0,2026-03-20,226.98,234.0,-3.0000,Sell,Buy,False
1,2026-03-23,224.83,223.0,0.8206,Buy,Buy,True
2,2026-03-24,222.93,216.5,2.9700,Buy,Buy,True
3,2026-03-25,208.37,226.5,-8.0044,Sell,Buy,False
4,2026-03-26,220.37,225.5,-2.2749,Sell,Hold,False
5,2026-03-27,214.23,219.5,-2.4009,Sell,Hold,False
6,2026-03-30,206.94,220.5,-6.1497,Sell,Hold,False
7,2026-03-31,202.49,198.5,2.0101,Buy,Hold,False
8,2026-04-01,214.70,211.0,1.7536,Buy,Hold,False
9,2026-04-02,220.00,200.5,9.7257,Liquidate,Sell,True


In [10]:
display(Markdown(f"![Phase 3 Signal Diagnostic]({phase3_signal_plot_path.relative_to(ROOT)})"))

![Phase 3 Signal Diagnostic](final_submission_artifacts/phase3_signal_return_comparison.png)

In [11]:
submission_trade_log, submission_metrics = build_submission_trade_log(
    actual_submission_df,
    amount_basis="predicted",
)
submission_trade_log_path = artifact_dir / "phase3_submission_trade_log.csv"
submission_trade_log.to_csv(submission_trade_log_path, index=False)

submission_asset_plot_path = artifact_dir / "phase3_submission_asset_curve.png"
save_asset_curve_plot(
    submission_trade_log,
    submission_asset_plot_path,
    "Phase 3 Actual Submission: Asset Curve (Transaction Amount By Predicted Price)",
)
phase3_position_plot_path = artifact_dir / "phase3_position_breakdown.png"
save_phase3_position_breakdown_plot(submission_trade_log, phase3_position_plot_path)

print(
    f"Submission final asset: {submission_metrics['final_total_asset']:.0f} TWD, "
    f"ROI: {submission_metrics['roi_pct']:.2f}%, "
    f"Max Drawdown: {submission_metrics['max_drawdown_pct']:.2f}%, "
    f"Max negative cash: {submission_metrics['max_negative_cash']:.0f} TWD"
)
submission_trade_log.round(4)

Submission final asset: 9986155 TWD, ROI: -0.14%, Max Drawdown: 11.96%, Max negative cash: 0 TWD


,Date,Sheet,Stock,PredictedPrice,CurrentClose,SubmittedAction,SubmittedQuantity,PricingBasis,PricingBasisPrice,TransactionAmount,CashBalance,HoldingsAfter,MarketValue,TotalAsset
0,2026-03-20,Day 1,2408,226.98,234.0,Buy,8547,predicted,226.98,1939998.06,8060001.94,8547,1999998.0,10059999.94
1,2026-03-23,Day 2,2408,224.83,223.0,Buy,13343,predicted,224.83,2999906.69,5060095.25,21890,4881470.0,9941565.25
2,2026-03-24,Day 3,2408,222.93,216.5,Buy,10000,predicted,222.93,2229300.00,2830795.25,31890,6904185.0,9734980.25
3,2026-03-25,Day 4,2408,208.37,226.5,Buy,12000,predicted,208.37,2500440.00,330355.25,43890,9941085.0,10271440.25
4,2026-03-26,Day 5,2408,220.37,225.5,Hold,0,predicted,220.37,0.00,330355.25,43890,9897195.0,10227550.25
5,2026-03-27,Day 6,2408,214.23,219.5,Hold,0,predicted,214.23,0.00,330355.25,43890,9633855.0,9964210.25
6,2026-03-30,Day 7,2408,206.94,220.5,Hold,0,predicted,206.94,0.00,330355.25,43890,9677745.0,10008100.25
7,2026-03-31,Day 8,2408,202.49,198.5,Hold,0,predicted,202.49,0.00,330355.25,43890,8712165.0,9042520.25
8,2026-04-01,Day 9,2408,214.70,211.0,Hold,0,predicted,214.70,0.00,330355.25,43890,9260790.0,9591145.25
9,2026-04-02,Day 10,2408,220.00,200.5,Sell,43890,predicted,220.00,9655800.00,9986155.25,0,0.0,9986155.25


In [12]:
display(Markdown(f"![Phase 3 Submission Asset Curve]({submission_asset_plot_path.relative_to(ROOT)})"))

![Phase 3 Submission Asset Curve](final_submission_artifacts/phase3_submission_asset_curve.png)

In [13]:
display(Markdown(f"![Phase 3 Position Breakdown]({phase3_position_plot_path.relative_to(ROOT)})"))

![Phase 3 Position Breakdown](final_submission_artifacts/phase3_position_breakdown.png)

In [14]:
rule_replay_df, rule_replay_metrics = build_rule_compliant_replay(actual_submission_df)
rule_replay_path = artifact_dir / "phase3_rule_compliant_replay.csv"
rule_replay_df.to_csv(rule_replay_path, index=False)

rule_replay_plot_path = artifact_dir / "phase3_rule_replay_asset_curve.png"
save_asset_curve_plot(
    rule_replay_df,
    rule_replay_plot_path,
    "Phase 3 Rule-Compliant Replay: Asset Curve (Close-Price Settlement)",
)
phase3_drawdown_plot_path = artifact_dir / "phase3_drawdown_comparison.png"
save_phase3_drawdown_comparison_plot(
    submission_trade_log,
    rule_replay_df,
    phase3_drawdown_plot_path,
)

print(
    f"Rule replay final asset: {rule_replay_metrics['final_total_asset']:.0f} TWD, "
    f"ROI: {rule_replay_metrics['roi_pct']:.2f}%, "
    f"Max Drawdown: {rule_replay_metrics['max_drawdown_pct']:.2f}%, "
    f"Clipped days: {rule_replay_metrics['clipped_trade_days']}"
)
rule_replay_df.round(4)

Rule replay final asset: 8941458 TWD, ROI: -10.59%, Max Drawdown: 12.19%, Clipped days: 0


,Date,Sheet,Stock,PredictedPrice,SettlementClose,RequestedAction,RequestedQuantity,ExecutedAction,ExecutedQuantity,TransactionAmount,CashBalance,HoldingsAfter,TotalAsset
0,2026-03-20,Day 1,2408,226.98,234.0,Buy,8547,Buy,8547,1999998.0,8000002.0,8547,10000000.0
1,2026-03-23,Day 2,2408,224.83,223.0,Buy,13343,Buy,13343,2975489.0,5024513.0,21890,9905983.0
2,2026-03-24,Day 3,2408,222.93,216.5,Buy,10000,Buy,10000,2165000.0,2859513.0,31890,9763698.0
3,2026-03-25,Day 4,2408,208.37,226.5,Buy,12000,Buy,12000,2718000.0,141513.0,43890,10082598.0
4,2026-03-26,Day 5,2408,220.37,225.5,Hold,0,Hold,0,0.0,141513.0,43890,10038708.0
5,2026-03-27,Day 6,2408,214.23,219.5,Hold,0,Hold,0,0.0,141513.0,43890,9775368.0
6,2026-03-30,Day 7,2408,206.94,220.5,Hold,0,Hold,0,0.0,141513.0,43890,9819258.0
7,2026-03-31,Day 8,2408,202.49,198.5,Hold,0,Hold,0,0.0,141513.0,43890,8853678.0
8,2026-04-01,Day 9,2408,214.70,211.0,Hold,0,Hold,0,0.0,141513.0,43890,9402303.0
9,2026-04-02,Day 10,2408,220.00,200.5,Sell,43890,Sell,43890,8799945.0,8941458.0,0,8941458.0


In [15]:
display(Markdown(f"![Phase 3 Rule Replay Asset Curve]({rule_replay_plot_path.relative_to(ROOT)})"))

![Phase 3 Rule Replay Asset Curve](final_submission_artifacts/phase3_rule_replay_asset_curve.png)

In [16]:
display(Markdown(f"![Phase 3 Drawdown Comparison]({phase3_drawdown_plot_path.relative_to(ROOT)})"))

![Phase 3 Drawdown Comparison](final_submission_artifacts/phase3_drawdown_comparison.png)